# Tutorial on `xarray.apply_ufunc` function

In this notebook I explore the functioning of function `xarray.apply_ufunc`. My goal is to apply an indicator to an xarray Dataset object cointaining the needed variables with their coordinates and additional coordinates for its optional parameters. In particular, data variables in this object do not depend on the additional coordinates, but the outcome of the computation is an xarray Dataset object with values depending on both the original coordinates and the additional coordinates, which become dimensions.

I do my tests with indicator [heat_wave_frequency](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_frequency) and I suppose to pass a dataset as input argument of the indicator (i.e. I specify `ds` instead of `thresh_tasmin` and `thresh_tasmin`).
I follow two roads:

1. I check the source code of indicators to understand how the management of `xarray.DataArray` objects as optional arguments is implemented, since in that case the evaluation is parallelised over the dimensions of those objects. My suspect is that some of the inner operations are broadcasted along the dimensions of the `xarray.DataArray` objects involved and I want to understand if that can be extended easily to my needs (e.g. if function `xarray.apply_ufunc` is used and how).
2. I do some experiments using `xclim.apply_ufunc`, learning from examples I found on the Internet and in the documentation.

Along point 1, I start to check [the source code of the indicator](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indicators/atmos/_temperature.py#L200). It is created by passing simple metadata, but the core of the implementation is the index used in calculations, which is [`xclim.indices.heat_wave_frequency`](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indices/_multivariate.py#L639).
This index works on the multidimensional objects through the function [`compare`](https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/indices/generic.py#L232) from module `xclim.indices.generic`. This function does some checks on its arguments and in the end it uses the functional form of the Python comparison operators (cfr. [`operator`](https://docs.python.org/3/library/operator.html) package), which are retrieved through function [`xarray.core.ops.get_op`](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/core/ops.py#L270).
When the binary operators are applied to `xarray.DataArray` objects, I think their overrided operators are called. In `xarray` some methods from `numpy` are not implemented directly but they are injected (i.e. this is done by [one of the parent classes](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/core/ops.py#L305) of `xarray.DataArray`), however I am not able to figure out where or how the basic comparison operators are implemented. Probably their generation is done automatically with some templates and starts with [this list](https://github.com/pydata/xarray/blob/cd3ab8d5580eeb3639d38e1e884d2d9838ef6aa1/xarray/util/generate_ops.py#L19).
Anyway this exploration is been useful because as long as I delegate the work to xarray functions which act on xclim DataArray objects then I am sure they are parallelised even if I do not know their implementation.

## References

Additional references other than the in-text links are:

- The [official documentation](https://docs.xarray.dev/en/stable/generated/xarray.apply_ufunc.html) is the starting point.
- An [official tutorial](https://tutorial.xarray.dev/advanced/apply_ufunc/apply_ufunc.html) on how to use the method is available.
- The [guide on applying unvectorized functions](https://docs.xarray.dev/en/stable/examples/apply_ufunc_vectorize_1d.html) is useful to extend the vectorization to the dimensions given by the parameters of the indicator.
- The [example on broadcasting](https://docs.xarray.dev/en/stable/user-guide/computation.html#broadcasting-by-dimension-name) in the documentation for main computations in xarray is useful to get the idea no the process.
- [Information on computation patterns](https://tutorial.xarray.dev/intermediate/01-high-level-computation-patterns.html) can be useful to tackle the problem with the best tools.
- In general [parallelisation](https://tutorial.xarray.dev/advanced/parallel-intro.html) is the objective.

## Set up

In [ ]:
import numpy  # Version 1.26.4.
import xarray  # Version 2024.5.0.
import xclim  # Version 0.49.0.

import utilities

In [ ]:
ds_hourly = utilities.open_dataset()
indicator = xclim.core.indicator.registry["HEAT_WAVE_FREQUENCY"]()

In [ ]:
ds_daily = xarray.Dataset()
_resample = ds_hourly["tas"].resample(time="1D")
ds_daily["tasmin"] = _resample.min()
ds_daily["tasmax"] = _resample.max()

## Create variations

Prepare storage for `xarray.DataArray` objects containing the values of varied parameters.

In [ ]:
# Only some parameters can be varied, they are identified among the properties
# listed at https://github.com/Ouranosinc/xclim/blob/edf1e13b7d47e4ead94506cdbb3bce1f977376f9/xclim/core/utils.py#L523.
_ALLOWED_KINDS = [
    xclim.core.utils.InputKind.QUANTIFIED,
    xclim.core.utils.InputKind.NUMBER
]

parameters = {}
for name_parameter, parameter in indicator._all_parameters.items():
    if parameter.kind in _ALLOWED_KINDS:
        # Prepare the storage for the xclim.DataArray objects
        # which will be the additional coordinates.
        parameters[name_parameter] = None

Create actual sets of values. They can transformed to `xarray.DataArray` objects.

In [ ]:
import pathlib
import sys

sys.path.append(str(pathlib.Path("..").resolve()))

from framework.ensemble.intervals import IntervalQuantity, IntervalNumeric

In [ ]:
intervals = {}

name = "thresh_tasmin"
intervals[name] = IntervalQuantity(
    left=xclim.units.units.Quantity("13.0 degC"),
    right=xclim.units.units.Quantity("45.0 degC"),
    name=name
)

name = "thresh_tasmax"
intervals[name] = IntervalQuantity(
    left=xclim.units.units.Quantity("3.0 degC"),
    right=xclim.units.units.Quantity("27.0 degC"),
    name=name
)

name = "window"
intervals[name] = IntervalNumeric(
    left=2,
    right=4,
    name=name
)

# Discretise the intervals using the same number of values for every parameter.
for name_parameter in parameters:
    interval = intervals[name_parameter]
    parameters[name_parameter] = interval.to_dataarray(
        num=3,
        name=interval.name,
        dims=interval.name
    )

## Evaluate indicator for a single combination of parameters

A reference for the outcomes is

In [ ]:
from framework.utils import to_scalar_quantity

q_thresh_tasmin = to_scalar_quantity(parameters["thresh_tasmin"][0], xclim.units.units)
q_thresh_tasmax = to_scalar_quantity(parameters["thresh_tasmax"][0], xclim.units.units)
q_window = to_scalar_quantity(parameters["window"][0], xclim.units.units)

ref = indicator(
    ds=ds_daily,
    thresh_tasmin=q_thresh_tasmin,
    thresh_tasmax=q_thresh_tasmax,
    window=q_window
)

and the equivalent operation using `xarray.apply_ufunc` is

In [ ]:
out = xarray.apply_ufunc(
    indicator,
    kwargs={
        "ds": ds_daily,
        "thresh_tasmin": q_thresh_tasmin,
        "thresh_tasmax": q_thresh_tasmax,
        "window": q_window
    },
    dask="allowed"
)
print(out.equals(ref))

## Evaluate indicator parallelising over the dimensions of parameters

First it is useful to identify the core dimensions for my use case, which are the dimensions used to perform the calculations.
In general, I should reinterpret the indicator evaluation as a  [split-apply-combine workflow](https://docs.xarray.dev/en/stable/user-guide/groupby.html) (also [a tutorial](https://tutorial.xarray.dev/fundamentals/03.2_groupby_with_xarray#grouped-computations) is available).

The indicator expects `tasmin` and `tasmax` inputs as `xarray.DataArray` objects with same shape and mandatory `time` dimension.
First data is preprocessed by comparing it to to arguments `thresh_tasmin` and `thresh_tasmax`, respectively. The workflow is applied to the resulting binary data.

Along `time` dimension a resampling is performed, i.e. from daily values to yearly values, and this is the split step: data is split in groups by their `time` coordinates, a group per year.
If `thresh_tasmin` and `thresh_tasmax` arguments are `xarray.DataArray` objects, then their dimensions are broadcasted to match the shape of the inputs and the following operations are replicated for each element of the arguments.

The apply step is skipped.

Finally, the combine step is just the count of sequences with specified length of consecutive days with temperatures above the specified thresholds. This is implemented with a run-lenght algorithm.

The split step could be placed after the apply-combine steps if `resample_before_rl` parameter is set to `False`. Anyway, this does not alter the shape of the output.

From this general understanding of the inner workings of the indicator, I can identify the following dimensions and parameters as core dimensions:

- `time`, because it is the dimension the resampling is applied on.

An immediate problem related to this list of core dimensions is the need to wrap the indicator function in a function which accepts the keyword parameters as positional parameters (e.g. by putting them in alfabetical order), otherwise dimension broadcasting does not work in `xarray.apply_ufunc`. Some code reflection is probably needed for this task to apply to generic indicators.

Core dimensions needs to be [specified in `input_core_dims` argument of `xarray.apply_ufunc`](https://tutorial.xarray.dev/advanced/apply_ufunc/core-dimensions.html#reducing-with-np-mean). Since `time` dimension changes size, it needs to be present also in [`output_core_dims` and `exclude_dims` arguments](https://tutorial.xarray.dev/advanced/apply_ufunc/complex-output-numpy.html#dimensions-that-change-size).

All other dimensions should be broadcasted, they are [called *broadcast dimensions* or *loop dimensions*](https://tutorial.xarray.dev/advanced/apply_ufunc/automatic-vectorizing-numpy.html#automatic-vectorization) because the function is applied to each of their elements equally. In particular, I need to loop over `thresh_tasmin`, `thresh_tasmax` and `window` dimensions and use their values as arguments for the indicator, while they are kept as output coordinates.

Another big problem is how xclim indicators and indices work: they accept quantified values for some parameters, i.e. numeric variables with information about their units. Instead, `xarray.apply_ufunc` does not allow to manipulate data other than numeric, hence data should be manipulated with the right units before its application.

In [ ]:
parameters["thresh_tasmin"] = xclim.units.convert_units_to(parameters["thresh_tasmin"], ds_daily["tasmin"])
parameters["thresh_tasmax"] = xclim.units.convert_units_to(parameters["thresh_tasmax"], ds_daily["tasmax"])

Also, it is clear the difference between adding dimensions for the parameters without relating these coordinates to the values as

In [ ]:
ds1 = ds_daily.assign_coords(parameters)
print(ds1)

and adding the parameters as broadcasted dimensions 

In [ ]:
ds_parameter = xarray.Dataset(coords=parameters)

ds2, _ = xarray.broadcast(ds_daily, ds_parameter)
print(ds2)

which allows to add the parameters values as coordinates, but increases the memory usage considerably because data are copied for each combination of the added coordinates.

The first option is better for memory efficiency, if it works.
The second option can be mitigated by setting chuncks explicitly for dask arrays.

In [ ]:
ds2 = ds2.chunk({f"{dim}": 1 for dim in parameters.keys()})